In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("housing.csv")
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [3]:
from sklearn.preprocessing import OneHotEncoder
cols=["ocean_proximity"]
ohe = OneHotEncoder(drop="first", sparse_output = False, handle_unknown = "ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(cols), index = df.index)
df = pd.concat([df.drop(columns = cols), encoded_df], axis = 1)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0.0,0.0,1.0,0.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0.0,0.0,1.0,0.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0.0,0.0,1.0,0.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0.0,0.0,1.0,0.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0.0,0.0,1.0,0.0


In [4]:
from sklearn.impute import SimpleImputer
missing_value_cols = ["total_bedrooms"]
imp_mean = SimpleImputer(strategy = "mean")
df[missing_value_cols] = imp_mean.fit_transform(df[missing_value_cols])

In [5]:
df.isnull().sum()

longitude                     0
latitude                      0
housing_median_age            0
total_rooms                   0
total_bedrooms                0
population                    0
households                    0
median_income                 0
median_house_value            0
ocean_proximity_INLAND        0
ocean_proximity_ISLAND        0
ocean_proximity_NEAR BAY      0
ocean_proximity_NEAR OCEAN    0
dtype: int64

In [6]:
x = df.drop("median_house_value",axis=1)
y = df["median_house_value"]

In [7]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [9]:
x_train_scaled

array([[ 1.27258656, -1.3728112 ,  0.34849025, ..., -0.01556621,
        -0.35564565,  2.62975816],
       [ 0.70916212, -0.87669601,  1.61811813, ..., -0.01556621,
        -0.35564565,  2.62975816],
       [-0.44760309, -0.46014647, -1.95271028, ..., -0.01556621,
        -0.35564565,  2.62975816],
       ...,
       [ 0.59946887, -0.75500738,  0.58654547, ..., -0.01556621,
        -0.35564565, -0.3802631 ],
       [-1.18553953,  0.90651045, -1.07984112, ..., -0.01556621,
        -0.35564565, -0.3802631 ],
       [-1.41489815,  0.99543676,  1.85617335, ..., -0.01556621,
         2.81178749, -0.3802631 ]], shape=(16512, 12))

In [10]:
x_train

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
14196,-117.03,32.71,33.0,3126.0,627.0,2300.0,623.0,3.2596,0.0,0.0,0.0,1.0
8267,-118.16,33.77,49.0,3382.0,787.0,1314.0,756.0,3.8125,0.0,0.0,0.0,1.0
17445,-120.48,34.66,4.0,1897.0,331.0,915.0,336.0,4.1563,0.0,0.0,0.0,1.0
14265,-117.11,32.69,36.0,1421.0,367.0,1418.0,355.0,1.9425,0.0,0.0,0.0,1.0
2271,-119.80,36.78,43.0,2382.0,431.0,874.0,380.0,3.5542,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
11284,-117.96,33.78,35.0,1330.0,201.0,658.0,217.0,6.3700,0.0,0.0,0.0,0.0
11964,-117.43,34.02,33.0,3084.0,570.0,1753.0,449.0,3.0500,1.0,0.0,0.0,0.0
5390,-118.38,34.03,36.0,2101.0,569.0,1756.0,527.0,2.9344,0.0,0.0,0.0,0.0
860,-121.96,37.58,15.0,3575.0,597.0,1777.0,559.0,5.7192,0.0,0.0,0.0,0.0


In [11]:
import torch
import torch.nn as nn
x_train_tensor = torch.tensor(x_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1,1)

x_test_tensor = torch.tensor(x_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1,1)

In [12]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [14]:
#Define our ANN Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model=nn.Sequential(
            #1st hidden layer
            nn.Linear(x_train.shape[1],256),
            nn.ReLU(),
            
            #2nd hidden layer
            nn.Linear(256,256),
            nn.ReLU(),
            
            
            nn.Linear(256,128),
            nn.ReLU(),
            #outputlayer
            nn.Linear(128,1),)
    def forward(self,x):
        return self.model(x)
        

In [15]:
import torch.optim as optim

model = ANN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [16]:
#Train the ANN
train_losses = []
val_loss = []
epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = loss.item()
    train_losses.append(epoch_train_loss)

    #validation
    running_val_loss = 0.0
    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs,yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss/len(test_loader)
    val_loss.append(epoch_val_loss)

    print(f"epoch{epoch+1}/{epochs} ==> train loss ={epoch_train_loss} & val loss = {epoch_val_loss}")

epoch1/100 ==> train loss =5463900160.0 & val loss = 6592625664.0
epoch2/100 ==> train loss =6452859392.0 & val loss = 5257723392.0
epoch3/100 ==> train loss =2489802752.0 & val loss = 4877448192.0
epoch4/100 ==> train loss =3029111552.0 & val loss = 4684510208.0
epoch5/100 ==> train loss =4381753344.0 & val loss = 4625264640.0
epoch6/100 ==> train loss =6693425664.0 & val loss = 4547745280.0
epoch7/100 ==> train loss =5857775616.0 & val loss = 4507676672.0
epoch8/100 ==> train loss =6797357568.0 & val loss = 4447736832.0
epoch9/100 ==> train loss =4910924288.0 & val loss = 4385473536.0
epoch10/100 ==> train loss =5730978816.0 & val loss = 4389730816.0
epoch11/100 ==> train loss =4525492224.0 & val loss = 4354267136.0
epoch12/100 ==> train loss =5499677184.0 & val loss = 4409098240.0
epoch13/100 ==> train loss =3873509120.0 & val loss = 4304554496.0
epoch14/100 ==> train loss =3534096640.0 & val loss = 4267133696.0
epoch15/100 ==> train loss =2181182720.0 & val loss = 4271091968.0
epoc

In [24]:
#Evaluation
model.eval()
with torch.no_grad():
    train_preds = model(x_train_tensor)
    test_preds = model(x_test_tensor)

    y_test_pred = test_preds.numpy().flatten()
    y_train_pred = train_preds.numpy().flatten()

    train_mse_loss = criterion(train_preds, y_train_tensor)
    test_mse_loss = criterion(test_preds, y_test_tensor)
y_test_actual = y_test.values.flatten() if hasattr(y_test, 'values') else y_test.flatten()
print("Training MSE:",train_mse_loss.item())
print("Testing MSE:",test_mse_loss.item())

from sklearn.metrics import r2_score
print("r2_score=",r2_score(y_test,test_preds))

Training MSE: 2789963520.0
Testing MSE: 3049451776.0
r2_score= 0.7672900773400048


In [25]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns=["predicted values"])
actual_df = pd.DataFrame(y_test.values,columns=["Actual Values"])
pd.concat([predicted_df,actual_df], axis=1)

,predicted values,Actual Values
0,46477.933594,47700.0
1,117665.578125,45800.0
2,484371.562500,500001.0
3,261093.203125,218600.0
4,289557.843750,278000.0
...,...,...
4123,205691.390625,263300.0
4124,243570.312500,266800.0
4125,461048.531250,500001.0
4126,74962.843750,72300.0
